# ML-09 - Validation and Research Claim Audit

This audit checks whether the capstone claims stay inside what the data and validation design can support.

## 1. Two paper findings + methodology questions

**Finding 1:** A learned model ranked future-decline candidates better than the visible-low-CTR rule on held-out clients.

Question: does the split prevent same-client leakage? **Yes:** the split holds out whole clients.

**Finding 2:** Position and CTR context were the strongest model signals in this run.

Question: does feature importance prove cause? **No:** it shows model reliance/association, not causal search-engine behavior.

In [1]:

from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Markdown, Image


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "work" / "scripts" / "run_capstone_analysis.py").exists():
            return candidate
    raise FileNotFoundError("Could not find repo root")

ROOT = find_repo_root(Path.cwd().resolve())
METRICS_PATH = ROOT / "work" / "outputs" / "capstone_metrics.json"
if not METRICS_PATH.exists():
    raise FileNotFoundError(
        "Missing work/outputs/capstone_metrics.json. Run `python work/scripts/run_capstone_analysis.py` from the repo root after Hugging Face login."
    )
metrics = json.loads(METRICS_PATH.read_text(encoding="utf-8"))
results = pd.DataFrame(metrics["results"])
top_features = pd.DataFrame(metrics["top_features"])
top20 = pd.DataFrame(metrics["top20_preview"])
print(f"Loaded metrics from {METRICS_PATH.relative_to(ROOT)}")
print(f"Eligible rows: {metrics['eligible_rows']:,}; clients: {metrics['eligible_clients']}; base rate: {metrics['base_rate']:.3f}")

display(results)
display(top_features.head(10))

Loaded metrics from work\outputs\capstone_metrics.json
Eligible rows: 24,547; clients: 18; base rate: 0.256


,method,roc_auc,avg_precision,precision_at_50,precision_at_100
0,logistic_regression,0.853186,0.729499,0.90,0.77
1,random_forest,0.871198,0.739586,0.90,0.78
2,baseline_visible_low_ctr,0.470689,0.258484,0.18,0.28


,feature,importance
0,num__avg_position_may,0.252179
1,num__avg_position_apr,0.203353
2,num__ctr_may,0.081764
3,num__imp_change_apr_to_may,0.063747
4,num__content_age_days,0.060400
5,num__clicks_may,0.058961
6,num__impressions_apr,0.048202
7,num__days_since_update,0.040416
8,num__word_count,0.035044
9,num__char_count,0.027729


## 2. My model under an honest split

The selected validation design is client holdout. I do not report a random-row split as the headline number because it could overstate performance when client-specific patterns repeat across pages.

In [2]:
audit = pd.DataFrame([{
    "validation_design": "client holdout",
    "held_out_clients": metrics["test_clients"],
    "held_out_rows": metrics["test_rows"],
    "base_rate": metrics["base_rate"],
}])
display(audit)

,validation_design,held_out_clients,held_out_rows,base_rate
0,client holdout,4,667,0.2555


## 3. Leakage audit

The model uses August/September features to predict October decline. October metrics are not used as features. IDs are used only for grouping, joining, and splitting. The starter-data leakage fields `trend_direction` and `trend_pct` are not used in this warehouse model.

In [3]:
display(pd.DataFrame({"leakage_check": metrics["leakage_exclusions"]}))

,leakage_check
0,"June metrics are label/output only, not model ..."
1,trend_direction and trend_pct are not warehous...
2,IDs are used for grouping/splitting only
3,validation holds out whole clients


## 4. Claim rewrite

Too strong: "The model proves which pages should be refreshed."

Careful version: "In this warehouse slice, the model ranked held-out client pages for future impression decline more effectively than the visible-low-CTR baseline; the output should be used as a decision-support queue for human review.

In [4]:
claim_table = pd.DataFrame([
    {"unsafe_wording": "proves which pages should be refreshed", "safe_wording": "ranked future-decline candidates for review"},
    {"unsafe_wording": "will recover traffic", "safe_wording": "looks worth reviewing before action"},
    {"unsafe_wording": "Google rewards these factors", "safe_wording": "these observed signals were useful to the model in this data"},
])
display(claim_table)

,unsafe_wording,safe_wording
0,proves which pages should be refreshed,ranked future-decline candidates for review
1,will recover traffic,looks worth reviewing before action
2,Google rewards these factors,these observed signals were useful to the mode...


## Self-check

- [x] Claims are observational and decision-support oriented.
- [x] Validation uses held-out clients.
- [x] Future-window and label-derived leakage risks are named.
- [x] No causal refresh-impact claim is made.